# ConvNeXt: five existing folds and OOF predictions

Keep `convnext_tiny.fb_in1k`, ImageNet-1k initialization, labels, full-frame transforms,
weighted training BCE, unweighted validation BCE, AdamW and the existing metric definitions.
Train one independent model per original fold; no split generation or relabeling.

The default imports the completed fold-0 `best.pt` and trains folds 1–4. Reuse requires
its original `validation_predictions.csv` next to it to verify the held-out IDs/labels.
Attach that training Output and the original `folds.csv` when starting a new Kaggle session.
To deliberately train fold 0 again, set `REUSE_FOLD0_PATH = None`.

Each fold exports predictions only for its held-out images, with and without horizontal
flip TTA. Once all five folds are complete, fit per-class thresholds to all OOF predictions.
OOF means each image's prediction comes from a model whose training set excluded that image
and its group. Checkpoint selection and threshold fitting still use held-out labels, so
the fitted OOF score is a calibration score, not an unbiased estimate of Kaggle performance.

This is a **new notebook**, not a reason to rerun `cnn_2.ipynb`. Expect roughly four more
training runs plus OOF inference when reusing fold 0. W&B creates one resumable run per new
fold. Completed folds are skipped. `last.pt` restores model/optimizer/scheduler/scaler at
the last completed epoch; `best.pt` is used for prediction. An interrupted epoch is repeated.

Outputs: `/kaggle/working/convnext_cv`. Save this entire directory and `folds.csv` before
ending the Kaggle session. For a new session, attach the saved Output and set
`PREVIOUS_CV_DIR` to its `convnext_cv` directory. Resume restores hyperparameter-compatible
states; it does not extend the original epoch budget. Keep the same package versions.

After training, run **cnn_2_ensemble.ipynb** to average test probabilities and export submissions.

W&B comparison: new runs use the shared team `yrachynska-kyiv-school-of-economics` / `cats-and-dogs`. They also log `val_f1`, `val_loss`, `train_loss_epoch` and `lr` against `epoch`, alongside the original `epoch_metrics/*` keys. For existing runs, select both `val_f1` and `epoch_metrics/val_f1` in a workspace line plot with X = `epoch`; old history is not rewritten. F1 aliases use the fixed 0.5 threshold, not the fitted calibration score. ResNet validation BCE is weighted and ConvNeXt validation BCE is unweighted, so their loss magnitudes are not directly comparable.

In [1]:
# Optional, separate install cell; restart the kernel afterwards.
# %pip install -q timm wandb "albumentations>=2,<3"

In [2]:
import json
import os
import random
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

print('torch:', torch.__version__, '| timm:', timm.__version__)
import hashlib
import shutil
from time import perf_counter

torch: 2.11.0+cu128 | timm: 1.0.29


## 1. Configuration

Keep the original folds. Use `FOLDS_TO_TRAIN = [1]` to train one new fold at a time; OOF aggregation waits until all five are present.

In [3]:
SEED = 21
MODEL_NAME = 'convnext_tiny.fb_in1k'
LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
DATA_DIR = Path('/kaggle/input/competitions/cats-and-dogs')
FOLDS_PATH = None
CV_DIR = Path('/kaggle/working/convnext_cv')
PREVIOUS_CV_DIR = None  # e.g. Path('/kaggle/input/my-saved-output/convnext_cv')
REUSE_FOLD0_PATH = Path('/kaggle/working/convnext_tiny_fb_in1k_fold0_resnet_pipeline/best.pt')
FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5
LR = 1e-4
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-2
THRESHOLD = 0.5
NUM_WORKERS = 0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
USE_WANDB = True
WANDB_PROJECT = 'cats-and-dogs'
WANDB_ENTITY = 'yrachynska-kyiv-school-of-economics'  # shared team project
WANDB_MODE = 'online'
WANDB_LOG_EVERY = 20
assert len(FOLDS_TO_TRAIN) == len(set(FOLDS_TO_TRAIN)) and set(FOLDS_TO_TRAIN).issubset(range(5))
assert EPOCHS > 0 and BATCH_SIZE > 0 and EARLY_STOPPING_PATIENCE > 0
assert 0 <= MIN_LR <= LR and NUM_WORKERS == 0
CV_DIR.mkdir(parents=True, exist_ok=True)
preprocessing = {'name': 'albumentations_resize_v1', 'image_size': IMAGE_SIZE,
                 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'interpolation': 1}
print('device:', DEVICE, '| requested folds:', FOLDS_TO_TRAIN, '| outputs:', CV_DIR)

device: cuda | requested folds: [0, 1, 2, 3, 4] | outputs: /kaggle/working/convnext_cv


In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

## 2. Load the original split

Validate IDs, labels, five-fold assignments and group separation. Fingerprint the exact CSV contents to reject stale checkpoints or changed splits.

In [5]:
def digest_text(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

def file_digest(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()

def save_json(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2))
    temporary.replace(path)

def save_torch(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)

if not ((DATA_DIR / 'train.csv').is_file() and (DATA_DIR / 'test.csv').is_file()):
    candidates = []
    for directory, subdirs, files in os.walk('/kaggle/input'):
        subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
        root = Path(directory)
        if {'train.csv', 'test.csv'}.issubset(files) and (root / 'train').is_dir() and (root / 'test').is_dir():
            if {'image_id', *LABEL_COLS}.issubset(pd.read_csv(root / 'train.csv', nrows=0).columns):
                candidates.append(root)
    if len(candidates) != 1:
        raise FileNotFoundError(f'Set DATA_DIR to the competition data. Candidates: {candidates}')
    DATA_DIR = candidates[0]
if FOLDS_PATH is None:
    paths = [Path('/kaggle/working/folds.csv'), Path('folds.csv'), CV_DIR / 'folds.csv']
    if PREVIOUS_CV_DIR is not None:
        paths.append(Path(PREVIOUS_CV_DIR) / 'folds.csv')
    candidates = {p.resolve() for p in paths if p.is_file()}
    if not candidates:
        for directory, subdirs, files in os.walk('/kaggle/input'):
            subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
            if 'folds.csv' in files:
                candidates.add((Path(directory) / 'folds.csv').resolve())
    # Copies with identical contents represent the same split.
    by_contents = {}
    for path in sorted(candidates):
        by_contents.setdefault(file_digest(path), path)
    if len(by_contents) != 1:
        raise FileNotFoundError(f'Attach the ORIGINAL folds.csv or set FOLDS_PATH. No split is generated. Candidates: {sorted(candidates)}')
    FOLDS_PATH = next(iter(by_contents.values()))
FOLDS_PATH = Path(FOLDS_PATH)
train = pd.read_csv(DATA_DIR / 'train.csv', dtype={'image_id': str})
folds = pd.read_csv(FOLDS_PATH, dtype={'image_id': str})
assert {'image_id', 'cluster', 'fold', *LABEL_COLS}.issubset(folds.columns)
assert train.image_id.notna().all() and train.image_id.is_unique
assert folds.image_id.notna().all() and folds.image_id.is_unique
assert set(train.image_id) == set(folds.image_id)
assert train[LABEL_COLS].isin([0, 1]).all().all()
aligned = folds.set_index('image_id').loc[train.image_id]
assert np.array_equal(train[LABEL_COLS].to_numpy(), aligned[LABEL_COLS].to_numpy())
assert folds[['cluster', 'fold']].notna().all().all()
assert set(folds.fold.unique()) == set(range(5))
assert folds.groupby('cluster').fold.nunique().eq(1).all(), 'A group crosses folds.'
frame = train.merge(folds[['image_id', 'cluster', 'fold']], on='image_id', validate='one_to_one', sort=False)
frame['path'] = frame.image_id.map(lambda ident: str(DATA_DIR / 'train' / f'{ident}.jpg'))
canonical = folds[['image_id', *LABEL_COLS, 'cluster', 'fold']].sort_values('image_id').copy()
canonical[['cluster', 'fold', *LABEL_COLS]] = canonical[['cluster', 'fold', *LABEL_COLS]].astype('int64')
folds_fingerprint = digest_text(canonical.to_csv(index=False, lineterminator=chr(10)))
cv_config = {
    'pipeline': 'convnext_cv_v1', 'model_name': MODEL_NAME, 'label_cols': LABEL_COLS,
    'versions': {'torch': str(torch.__version__), 'timm': str(timm.__version__), 'albumentations': str(A.__version__)},
    'folds_fingerprint': folds_fingerprint, 'seed': SEED, 'epochs': EPOCHS,
    'batch_size': BATCH_SIZE, 'lr': LR, 'min_lr': MIN_LR, 'weight_decay': WEIGHT_DECAY,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE, 'preprocessing': preprocessing,
    'train_loss': 'BCE_sqrt_pos_weight', 'val_loss': 'BCE_unweighted', 'threshold': THRESHOLD,
}
cv_fingerprint = digest_text(json.dumps(cv_config, sort_keys=True))
cv_config['cv_fingerprint'] = cv_fingerprint
config_path = CV_DIR / 'cv_config.json'
if config_path.is_file():
    assert json.loads(config_path.read_text()) == cv_config, 'CV settings changed; use a new CV_DIR.'
if PREVIOUS_CV_DIR is not None:
    PREVIOUS_CV_DIR = Path(PREVIOUS_CV_DIR)
    previous_config = json.loads((PREVIOUS_CV_DIR / 'cv_config.json').read_text())
    assert previous_config == cv_config, 'Previous CV output uses different settings or folds.'
    # Carry all saved folds forward even when training only one requested fold now.
    for previous_f in range(5):
        source_folder = PREVIOUS_CV_DIR / f'fold_{previous_f}'
        destination = CV_DIR / f'fold_{previous_f}'
        if not any((destination / name).exists() for name in ['best.pt', 'last.pt', 'complete.json']):
            destination.mkdir(parents=True, exist_ok=True)
            for name in ['best.pt', 'last.pt', 'history.csv', 'oof.csv', 'complete.json', 'wandb_id.txt']:
                if (source_folder / name).is_file():
                    shutil.copy2(source_folder / name, destination / name)

save_json(cv_config, config_path)
folds.to_csv(CV_DIR / 'folds.csv', index=False)
print('Original folds:', FOLDS_PATH, '| fingerprint:', folds_fingerprint[:12])
display(frame.groupby('fold')[LABEL_COLS].sum())

Original folds: /kaggle/working/folds.csv | fingerprint: 453ed1d2e114


,Cat,Dog,Moris,Motya,Biatrix
fold,,,,,
0,1006,982,14,7,9
1,1027,1007,12,7,7
2,1024,1002,12,14,5
3,1031,1003,7,4,14
4,989,1031,11,10,6


## 3. Existing transforms, losses and validation

The augmentation recipe and validation function are reused. Epoch selection stays at 0.5. At each epoch boundary, reseed the augmentation and shuffle generator so resuming does not depend on missing in-memory RNG states.

In [6]:
def make_transforms():
    train_transform = A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=preprocessing['interpolation']),
        A.HorizontalFlip(p=0.5),
        A.Affine(translate_percent=(-0.1, 0.1), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
        A.Normalize(mean=preprocessing['mean'], std=preprocessing['std']),
        ToTensorV2(),
    ])
    eval_transform = A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=preprocessing['interpolation']),
        A.Normalize(mean=preprocessing['mean'], std=preprocessing['std']),
        ToTensorV2(),
    ])
    return train_transform, eval_transform

class PetDataset(Dataset):
    def __init__(self, frame, transform, with_labels=True):
        self.paths = frame.path.tolist()
        self.targets = frame[LABEL_COLS].to_numpy(dtype=np.float32) if with_labels else None
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            pixels = np.array(image.convert('RGB'))
        x = self.transform(image=pixels)['image']
        if self.targets is None:
            return x
        return x, torch.from_numpy(self.targets[index])

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(frame, training=False, with_labels=True):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        PetDataset(frame, train_transform if training else eval_transform, with_labels),
        batch_size=BATCH_SIZE, shuffle=training, num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda', drop_last=False,
        worker_init_fn=seed_worker, generator=generator,
    )

In [7]:
def metric_report(targets, probabilities, threshold=THRESHOLD):
    predictions = (probabilities >= threshold).astype(np.int32)
    rows = []
    for j, label in enumerate(LABEL_COLS):
        y = targets[:, j]
        auc = roc_auc_score(y, probabilities[:, j]) if np.unique(y).size == 2 else np.nan
        rows.append({
            'label': label,
            'positives': int(y.sum()),
            'predicted positives': int(predictions[:, j].sum()),
            'f1': f1_score(y, predictions[:, j], zero_division=0),
            'precision': precision_score(y, predictions[:, j], zero_division=0),
            'recall': recall_score(y, predictions[:, j], zero_division=0),
            'roc_auc': auc,
        })
    table = pd.DataFrame(rows).set_index('label')
    valid_auc = table.roc_auc.dropna()
    metrics = {
        'macro_f1': float(f1_score(targets, predictions, average='macro', zero_division=0)),
        'macro_roc_auc': float(valid_auc.mean()) if len(valid_auc) else float('nan'),
    }
    return metrics, table

def tune_thresholds(targets, probabilities):
    thresholds = np.full(len(LABEL_COLS), THRESHOLD, dtype=np.float64)
    candidates = np.round(np.linspace(0.05, 0.95, 91), 2)
    for j, label in enumerate(LABEL_COLS):
        if np.unique(targets[:, j]).size < 2:
            print(f'{label}: keeping {THRESHOLD}; validation does not contain both outcomes.')
            continue
        scores = np.array([
            f1_score(targets[:, j], probabilities[:, j] >= t, zero_division=0)
            for t in candidates
        ])
        tied = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-12))
        best = tied[np.argmin(np.abs(candidates[tied] - THRESHOLD))]
        thresholds[j] = candidates[best]
    return thresholds

In [8]:
def train_one_epoch():
    global train_step
    model.train()
    total_loss = 0.0
    seen = 0
    progress = tqdm(train_loader, desc='train', leave=False)
    for batch_index, (images, targets) in enumerate(progress, start=1):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
            loss = train_criterion(logits, targets)
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Non-finite training loss: {loss.item()}')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(images)
        seen += len(images)
        progress.set_postfix(loss=f'{total_loss / seen:.4f}')
        train_step += 1
        if wandb_run is not None and (train_step % WANDB_LOG_EVERY == 0 or batch_index == len(train_loader)):
            wandb_run.log({
                'train_step': train_step, 'batch/loss': loss.item(),
                'batch/running_loss': total_loss / seen,
            })
    return total_loss / seen

@torch.inference_mode()
def validate():
    model.eval()
    total_loss = 0.0
    all_targets, all_probabilities = [], []
    for images, targets in tqdm(val_loader, desc='validation', leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
        loss = val_criterion(logits.float(), targets)
        total_loss += loss.item() * len(images)
        all_targets.append(targets.cpu().numpy())
        all_probabilities.append(logits.float().sigmoid().cpu().numpy())
    targets = np.concatenate(all_targets)
    probabilities = np.concatenate(all_probabilities)
    assert np.isfinite(probabilities).all(), 'Non-finite validation probabilities.'
    metrics, table = metric_report(targets, probabilities)
    metrics['val_loss'] = total_loss / len(val_df)
    assert np.isfinite(metrics['val_loss']), 'Non-finite validation loss.'
    return metrics, table, probabilities

In [9]:
wandb_run = None
train_step = 0

def start_wandb_run(config):
    if not USE_WANDB or WANDB_MODE == 'disabled':
        return nullcontext(None)
    if WANDB_MODE not in {'online', 'offline'}:
        raise ValueError('WANDB_MODE must be online, offline, or disabled.')
    if WANDB_LOG_EVERY < 1:
        raise ValueError('WANDB_LOG_EVERY must be positive.')
    try:
        import wandb
    except ImportError:
        raise ImportError('Install wandb with %pip install -q wandb, or set USE_WANDB = False.') from None

    if WANDB_MODE == 'online' and not os.environ.get('WANDB_API_KEY'):
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
        except Exception:
            raise RuntimeError(
                'Add the WANDB_API_KEY Kaggle Secret and grant notebook access, '
                'or set the WANDB_API_KEY environment variable. '
                'For local logging use WANDB_MODE = "offline".'
            ) from None
        if not api_key:
            raise RuntimeError('WANDB_API_KEY is empty. Check the Kaggle Secret.')
        os.environ['WANDB_API_KEY'] = api_key  # never print or put the key in run config

    id_path = OUTPUT_DIR / 'wandb_id.txt'
    run_id = id_path.read_text().strip() if id_path.is_file() else wandb.util.generate_id()
    id_path.write_text(run_id)
    return wandb.init(
        id=run_id, resume='allow', group=f'convnext-cv-{cv_fingerprint[:8]}',
        project=WANDB_PROJECT, entity=WANDB_ENTITY, name=WANDB_RUN_NAME,
        mode=WANDB_MODE, config=config, dir=str(OUTPUT_DIR), save_code=False,
    )

In [10]:
@torch.inference_mode()
def predict_val_pair():
    model.eval()
    originals, averages = [], []
    for images, _ in tqdm(val_loader, desc=f'OOF fold {FOLD}: original + flip'):
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
        original = logits.float().sigmoid()
        del logits
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits_flip = model(torch.flip(images, dims=[3]))
        originals.append(original.cpu().numpy())
        averages.append(((original + logits_flip.float().sigmoid()) * 0.5).cpu().numpy())
    return np.concatenate(originals), np.concatenate(averages)

def check_oof(path, held_out):
    result = pd.read_csv(path, dtype={'image_id': str})
    assert result.image_id.is_unique and set(result.image_id) == set(held_out.image_id)
    result = result.set_index('image_id').loc[held_out.image_id].reset_index()
    assert np.array_equal(result[LABEL_COLS].to_numpy(), held_out[LABEL_COLS].to_numpy())
    assert np.array_equal(result['fold'].to_numpy(), held_out['fold'].to_numpy())
    assert np.array_equal(result['cluster'].to_numpy(), held_out['cluster'].to_numpy())
    for suffix in ('_prob', '_prob_tta'):
        p = result[[label + suffix for label in LABEL_COLS]].to_numpy()
        assert p.shape == (len(held_out), len(LABEL_COLS)) and np.isfinite(p).all()
        assert ((p >= 0) & (p <= 1)).all()
    return result

## 4. Train/reuse/resume folds

Never initialize folds 1–4 from fold-0 trained weights: that would expose their validation
images during pretraining. Every new fold starts independently with `pretrained=True`.
Reused or resumed complete model states are restored with `pretrained=False`.

Each epoch writes `last.pt` atomically with optimizer/scheduler/scaler state. Completed
folds have `complete.json`, the best checkpoint hash and validated OOF rows. Downloading
only `best.pt` is sufficient for inference, but not for exact training continuation.

In [11]:
for FOLD in FOLDS_TO_TRAIN:
    OUTPUT_DIR = CV_DIR / f'fold_{FOLD}'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    BEST_PATH = OUTPUT_DIR / 'best.pt'
    LAST_PATH = OUTPUT_DIR / 'last.pt'
    complete_path = OUTPUT_DIR / 'complete.json'
    oof_path = OUTPUT_DIR / 'oof.csv'
    train_df = frame.loc[frame.fold != FOLD].reset_index(drop=True)
    val_df = frame.loc[frame.fold == FOLD].reset_index(drop=True)
    assert len(train_df) > 0 and len(val_df) > 0
    assert set(train_df.cluster).isdisjoint(val_df.cluster)
    if complete_path.is_file():
        done = json.loads(complete_path.read_text())
        assert done['cv_fingerprint'] == cv_fingerprint and done['fold'] == FOLD
        assert done['checkpoint_sha256'] == file_digest(BEST_PATH), 'Changed completed checkpoint.'
        check_oof(oof_path, val_df)
        print(f'Fold {FOLD}: completed; skipping training and OOF inference.')
        continue

    seed_everything(SEED + FOLD)
    fold_config = {**cv_config, 'fold': FOLD, 'folds_path': str(FOLDS_PATH),
                   'image_size': IMAGE_SIZE, 'num_classes': len(LABEL_COLS)}
    imported = None
    if FOLD == 0 and REUSE_FOLD0_PATH is not None and not BEST_PATH.is_file() and not LAST_PATH.is_file():
        source_path = Path(REUSE_FOLD0_PATH)
        if not source_path.is_file():
            candidates = []
            for directory, subdirs, files in os.walk('/kaggle/input'):
                subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
                if Path(directory).name == 'convnext_tiny_fb_in1k_fold0_resnet_pipeline' and 'best.pt' in files:
                    candidates.append(Path(directory) / 'best.pt')
            if len(candidates) != 1:
                raise FileNotFoundError('Attach the original fold-0 Output, set REUSE_FOLD0_PATH, or set it to None to retrain fold 0.')
            source_path = candidates[0]
        imported = torch.load(source_path, map_location='cpu', weights_only=True)
        original_config = imported['config']
        assert original_config['model_name'] == MODEL_NAME and original_config['label_cols'] == LABEL_COLS
        assert original_config['fold'] == FOLD and original_config.get('preprocessing') == preprocessing
        proof_path = source_path.parent / 'validation_predictions.csv'
        if not proof_path.is_file():
            raise FileNotFoundError('Fold-0 reuse needs its original validation_predictions.csv next to best.pt to verify the split. Attach that Output or set REUSE_FOLD0_PATH = None.')
        proof = pd.read_csv(proof_path, dtype={'image_id': str})
        assert proof.image_id.is_unique and set(proof.image_id) == set(val_df.image_id), 'Fold-0 validation IDs differ.'
        proof = proof.set_index('image_id').loc[val_df.image_id]
        assert np.array_equal(proof[LABEL_COLS].to_numpy(), val_df[LABEL_COLS].to_numpy()), 'Fold-0 validation labels differ.'
        assert (proof.fold == FOLD).all()
        fold_config['source_checkpoint'] = str(source_path)
        save_torch({'model': imported['model'], 'epoch': imported['epoch'], 'metrics': imported['metrics'],
                    'config': fold_config, 'imported_config': original_config}, BEST_PATH)
        print('Reusing completed fold 0:', source_path)
    last = torch.load(LAST_PATH, map_location='cpu', weights_only=True) if LAST_PATH.is_file() else None
    if last is not None:
        assert last['config']['cv_fingerprint'] == cv_fingerprint and last['config']['fold'] == FOLD
    # A best-only interrupted folder cannot safely resume training without last.pt.
    if BEST_PATH.is_file() and last is None and imported is None:
        existing = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
        assert existing['config']['cv_fingerprint'] == cv_fingerprint and existing['config']['fold'] == FOLD
        if 'source_checkpoint' in existing['config']:
            imported = existing  # interrupted after importing a completed fold
        else:
            raise RuntimeError(f'Fold {FOLD}: best.pt exists without last.pt or completion marker. Restore last.pt from saved Output, or use a fresh CV_DIR.')

    model = timm.create_model(MODEL_NAME, pretrained=(last is None and imported is None), num_classes=len(LABEL_COLS)).to(DEVICE)
    data_config = timm.data.resolve_data_config({}, model=model)
    assert np.allclose(data_config['mean'], preprocessing['mean']) and np.allclose(data_config['std'], preprocessing['std'])
    train_transform, eval_transform = make_transforms()
    train_loader = make_loader(train_df, training=True)
    val_loader = make_loader(val_df)
    positive_counts = train_df[LABEL_COLS].sum().to_numpy(dtype=np.float32)
    negative_counts = len(train_df) - positive_counts
    assert (positive_counts > 0).all() and (negative_counts > 0).all()
    pos_weight = torch.as_tensor(np.sqrt(negative_counts / positive_counts), device=DEVICE)
    train_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    val_criterion = nn.BCEWithLogitsLoss()
    fold_config['pos_weight'] = pos_weight.cpu().tolist()
    fold_config['data_config'] = {**data_config, 'input_size': (3, IMAGE_SIZE, IMAGE_SIZE)}
    if imported is None:
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
        scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
        history, best_f1, best_loss, patience, start_epoch, train_step = [], -1.0, float('inf'), 0, 1, 0
        training_complete = False
        if last is not None:
            model.load_state_dict(last['model'])
            optimizer.load_state_dict(last['optimizer'])
            scheduler.load_state_dict(last['scheduler'])
            scaler.load_state_dict(last['scaler'])
            history = last['history']
            best_f1, best_loss, patience = last['best_f1'], last['best_loss'], last['patience']
            start_epoch, train_step = last['epoch'] + 1, last['train_step']
            training_complete = last['training_complete']
            print(f'Fold {FOLD}: resume after epoch {last["epoch"]}.')
            assert BEST_PATH.is_file(), 'Resume needs both best.pt and last.pt.'
            del last
        WANDB_RUN_NAME = f'convnext-cv-fold{FOLD}'
        wandb_run = None
        try:
            if not training_complete:
                with start_wandb_run(fold_config) as wandb_run:
                    if wandb_run is not None:
                        wandb_run.define_metric('train_step')
                        wandb_run.define_metric('batch/*', step_metric='train_step')
                        wandb_run.define_metric('epoch')
                        wandb_run.define_metric('epoch_metrics/*', step_metric='epoch')
                        wandb_run.define_metric('epoch_metrics/val_f1', step_metric='epoch', summary='max')
                        # Shared names match the ResNet runs; keep scalar last-value summaries.
                        for metric in ('val_f1', 'val_loss', 'train_loss_epoch', 'lr'):
                            wandb_run.define_metric(metric, step_metric='epoch')
                        print('W&B:', wandb_run.url if WANDB_MODE == 'online' else wandb_run.dir)
                    for epoch in range(start_epoch, EPOCHS + 1):
                        epoch_seed = SEED + FOLD * 1000 + epoch
                        seed_everything(epoch_seed)
                        if hasattr(train_transform, 'set_random_seed'):
                            train_transform.set_random_seed(epoch_seed)
                        train_loader.generator.manual_seed(epoch_seed)
                        started = perf_counter()
                        epoch_lr = optimizer.param_groups[0]['lr']
                        train_loss = train_one_epoch()
                        metrics, table, probabilities = validate()
                        targets = val_df[LABEL_COLS].to_numpy(dtype=np.int32)
                        metrics['val_accuracy'] = float(((probabilities >= THRESHOLD) == targets).all(axis=1).mean())
                        improved = metrics['macro_f1'] > best_f1
                        patience = 0 if improved else patience + 1
                        row = {'epoch': epoch, 'train_loss': train_loss, **metrics, 'lr': epoch_lr,
                               'epoch_seconds': perf_counter() - started, 'epochs_without_improvement': patience}
                        history.append(row)
                        if improved or (metrics['macro_f1'] == best_f1 and metrics['val_loss'] < best_loss):
                            best_f1, best_loss = metrics['macro_f1'], metrics['val_loss']
                            save_torch({'model': model.state_dict(), 'epoch': epoch, 'metrics': metrics, 'config': fold_config}, BEST_PATH)
                        scheduler.step()
                        training_complete = epoch == EPOCHS or patience >= EARLY_STOPPING_PATIENCE
                        save_torch({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                                    'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                                    'epoch': epoch, 'train_step': train_step, 'history': history,
                                    'best_f1': best_f1, 'best_loss': best_loss, 'patience': patience,
                                    'training_complete': training_complete, 'config': fold_config}, LAST_PATH)
                        pd.DataFrame(history).to_csv(OUTPUT_DIR / 'history.csv', index=False)
                        print(f'Fold {FOLD} | epoch {epoch}/{EPOCHS} | train {train_loss:.5f} | val {metrics["val_loss"]:.5f} | F1 {metrics["macro_f1"]:.5f} | patience {patience}/{EARLY_STOPPING_PATIENCE}')
                        if wandb_run is not None:
                            logs = {'epoch': epoch, 'epoch_metrics/val_f1': metrics['macro_f1']}
                            logs.update({f'epoch_metrics/{k}': v for k, v in row.items() if k != 'epoch' and np.isfinite(v)})
                            logs.update({f'epoch_metrics/f1_{label}': float(table.loc[label, 'f1']) for label in LABEL_COLS})
                            # Compatibility aliases for side-by-side ResNet/ConvNeXt panels.
                            logs.update({'val_f1': metrics['macro_f1'], 'val_loss': metrics['val_loss'],
                                         'train_loss_epoch': train_loss, 'lr': epoch_lr})
                            wandb_run.log(logs)
                            wandb_run.summary.update({'best/macro_f1': best_f1, 'completed_epochs': epoch})
                        if training_complete:
                            break
        finally:
            wandb_run = None
    best_checkpoint = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
    model.load_state_dict(best_checkpoint['model'])
    originals, averages = predict_val_pair()
    result = val_df[['image_id', 'cluster', 'fold', *LABEL_COLS]].copy()
    for j, label in enumerate(LABEL_COLS):
        result[label + '_prob'] = originals[:, j]
        result[label + '_prob_tta'] = averages[:, j]
    result.to_csv(oof_path, index=False)
    check_oof(oof_path, val_df)
    save_json({'fold': FOLD, 'cv_fingerprint': cv_fingerprint,
               'checkpoint_sha256': file_digest(BEST_PATH), 'best_epoch': best_checkpoint['epoch'],
               'validation_images': len(val_df)}, complete_path)
    print(f'Fold {FOLD}: completed and OOF saved.')
    for name in ('model', 'optimizer', 'scheduler', 'scaler', 'last', 'imported', 'existing', 'best_checkpoint'):
        globals().pop(name, None)
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

Reusing completed fold 0: /kaggle/working/convnext_tiny_fb_in1k_fold0_resnet_pipeline/best.pt


OOF fold 0: original + flip: 100%|██████████| 77/77 [00:26<00:00,  2.90it/s]


Fold 0: completed and OOF saved.


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mkovalenko (mkovalenko-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B: https://wandb.ai/mkovalenko-kyiv-school-of-economics/cats-and-dogs/runs/ev0h1x6a


...

Fold 4 | epoch 14/15 | train 0.00024 | val 0.01233 | F1 0.96672 | patience 4/5


Fold 4 | epoch 15/15 | train 0.00024 | val 0.01238 | F1 0.96662 | patience 5/5


batch/loss,▂▂▁▁▁▄█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/running_loss,█▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
epoch_metrics/epoch_seconds,█▃▂▂▁▁▁▁▁▁▂▂▁▁▂
epoch_metrics/epochs_without_improvement,▁▁▁▁▂▄▅▁▁▁▂▄▅▇█
epoch_metrics/f1_Biatrix,▁█▇██▇█████████
epoch_metrics/f1_Cat,▁▁▇▇▇██▆▇█▇▇▇▇▇
epoch_metrics/f1_Dog,▁▆██▅██▅▇▇▇▇▇▇▇
epoch_metrics/f1_Moris,▁▁▅█▇▆▅████████
epoch_metrics/f1_Motya,▅▁█▇▇█▇▇▇██████
+8,...


OOF fold 4: original + flip: 100%|██████████| 77/77 [00:15<00:00,  4.88it/s]


Fold 4: completed and OOF saved.


## 5. Full OOF calibration

Require all five completed folds, exactly one held-out prediction per training image.
Do not average predictions of other fold models on that image: those models trained on it.
Thresholds are fitted separately for original and TTA OOF probabilities. Test inference
will average the five models' probabilities and use these OOF-derived thresholds. The
single-model OOF and five-model test probability distributions may differ; evaluate the
result on Kaggle. This notebook does not submit anything automatically.

In [12]:
missing_folds = [f for f in range(5) if not (CV_DIR / f'fold_{f}' / 'complete.json').is_file()]
if missing_folds:
    print('OOF calibration not ready. Train/import remaining folds:', missing_folds)
else:
    parts = []
    checkpoint_hashes = {}
    for f in range(5):
        folder = CV_DIR / f'fold_{f}'
        done = json.loads((folder / 'complete.json').read_text())
        assert done['cv_fingerprint'] == cv_fingerprint and done['fold'] == f
        assert done['checkpoint_sha256'] == file_digest(folder / 'best.pt')
        checkpoint_hashes[str(f)] = done['checkpoint_sha256']
        held_out = frame.loc[frame.fold == f].reset_index(drop=True)
        parts.append(check_oof(folder / 'oof.csv', held_out))
    oof = pd.concat(parts, ignore_index=True)
    assert oof.image_id.is_unique and set(oof.image_id) == set(frame.image_id)
    oof = oof.set_index('image_id').loc[frame.image_id].reset_index()
    targets = frame[LABEL_COLS].to_numpy(dtype=np.int32)
    all_thresholds, reports = {}, []
    for method, suffix in [('original', '_prob'), ('tta', '_prob_tta')]:
        probabilities = oof[[label + suffix for label in LABEL_COLS]].to_numpy()
        thresholds = tune_thresholds(targets, probabilities)
        all_thresholds[method] = thresholds.tolist()
        for label, cutoffs in [('fixed_0_5', 0.5), ('fitted_oof', thresholds)]:
            report, _ = metric_report(targets, probabilities, cutoffs)
            report['val_accuracy'] = float(((probabilities >= cutoffs) == targets).all(axis=1).mean())
            reports.append({'method': method, 'threshold_source': label, **report})
    summary = pd.DataFrame(reports)
    display(summary)
    oof.to_csv(CV_DIR / 'oof_predictions.csv', index=False)
    summary.to_csv(CV_DIR / 'oof_metrics.csv', index=False)
    save_json({'model_name': MODEL_NAME, 'label_cols': LABEL_COLS, 'folds': list(range(5)),
               'cv_fingerprint': cv_fingerprint, 'folds_fingerprint': folds_fingerprint,
               'preprocessing': preprocessing, 'thresholds': all_thresholds,
               'checkpoint_hashes': checkpoint_hashes,
               'aggregation': 'equal_mean_sigmoid_probabilities', 'tta': 'original_and_horizontal_flip',
               'threshold_source': 'all_held_out_oof_predictions'}, CV_DIR / 'ensemble_config.json')
    print('Ready: run cnn_2_ensemble.ipynb with CV_DIR =', CV_DIR)

,method,threshold_source,macro_f1,macro_roc_auc,val_accuracy
0,original,fixed_0_5,0.976454,0.999723,0.992333
1,original,fitted_oof,0.978271,0.999723,0.992741
2,tta,fixed_0_5,0.969678,0.999758,0.992251
3,tta,fitted_oof,0.974188,0.999758,0.992496


Ready: run cnn_2_ensemble.ipynb with CV_DIR = /kaggle/working/convnext_cv
